---
tags: [algorithm, chemistry, resource-estimation, simulation]
---

# FTQC化学計算のリソース推定

このノートブックでは、Qamomileでfault-tolerantな量子化学計算のリソースモデルを比較する方法を示します。完全な論理回路がまだない段階で重要になる、Hamiltonian normalization、phase-estimation iterations、ToffoliまたはT counts、論理量子ビット、物理量子ビット、runtime proxiesに注目します。

In [1]:
# 最新のQamomileをpipでインストールします！
# !pip install qamomile

In [2]:
import sympy as sp
from openfermion import QubitOperator

import qamomile.observable as qm_o
from qamomile.circuit.estimator.algorithmic import (
    ChemistryQPEMethod,
    ChemistryQPEModel,
    FTQCResourceQuantity,
    SurfaceCodeDistanceBudget,
    block_encoding_from_chemistry_model,
    compare_ftqc_resource_estimates,
    estimate_qubitized_chemistry_qpe_from_model,
    estimate_qubitized_qpe_from_block_encoding,
    estimate_single_ancilla_trotter_qpe_from_hamiltonian,
    iter_ftqc_resource_quantity_specs,
    summarize_ftqc_resource_comparison,
    summarize_openfermion_qubit_operator,
    summarize_pauli_hamiltonian,
)

## Background

Fault-tolerantな化学計算アルゴリズムは、NISQのvariationalな例とは異なるリソース量で比較されることが多くあります。qubitized QPEでは、Hamiltonian block-encoding normalizationがwalk callsの回数を決め、per-walk implementationがToffoli countを決めます。近年の化学計算向け提案では、backend-levelのgate decompositionだけを変えるのではなく、Hamiltonian representationを変えることでこれらのコストを下げようとします。

例として、symmetry-compressed double factorization([arXiv:2403.03502](https://arxiv.org/abs/2403.03502))、simultaneous symmetry shifts and tensor factorizations([arXiv:2412.01338](https://arxiv.org/abs/2412.01338))、unitary weight concentrationを使うearly-FTQC single-ancilla QPE([arXiv:2603.22778](https://arxiv.org/abs/2603.22778))があります。以下のEstimatorは意図的にsymbolicです。具体的なHamiltonian-loading circuitに進む前に、提案したrepresentationがコストを支配する量をどう変えるかを確認できます。

## Problem Settings

同じactive-space scaleについて、次の3つのシナリオを比較します。

1. tensor-hypercontraction-likeなqubitized QPE baseline、
2. symmetry-compressed double-factorization-styleなqubitized QPE estimate、
3. unitary-weight concentration factorを持つearly-FTQC single-ancilla Trotter QPE estimate。

数値は小さく、人工的に選んでいます。このノートブックを高速でレビューしやすくするためです。特定の分子についての主張ではなく、workflowのデモとして読んでください。

In [3]:
n_spin_orbitals = 40
precision = sp.Float("0.0016")  # Hartree単位のおおよそのchemical accuracy

distance_budget = SurfaceCodeDistanceBudget(
    physical_error_rate=sp.Float("1e-3"),
    threshold_error_rate=sp.Float("1e-2"),
    target_logical_failure_probability=sp.Float("1e-9"),
    logical_operation_budget=1000,
)
architecture = distance_budget.to_surface_code_cost_model(
    physical_cycle_time_seconds=sp.Float("5e-8"),
    physical_qubits_per_logical_factor=2,
    logical_cycle_factor=1,
    factory_count=4,
    physical_qubits_per_factory=5000,
    factory_cycles_per_toffoli=2,
)
cost_model = architecture

assert distance_budget.code_distance == 21
assert architecture.physical_qubits_per_logical == 882
assert architecture.factory_qubits == 20000

## Resource Quantities

symbolicなFTQC estimatesでは、problem quantities、logical work、physical
assumptionsを分けて扱うべきです。Qamomileはcanonical quantity catalogを公開
しているので、downstream reportsは安定したkeyを使いつつ、読者向けのlabel、
unit、modeling layerも表示できます。

In [4]:
quantity_catalog = [
    spec.to_dict()
    for spec in iter_ftqc_resource_quantity_specs()
    if spec.quantity.value
    in {
        "lambda_norm",
        "target_precision",
        "qpe_iterations",
        "toffoli_gates",
        "t_gates",
        "logical_qubits",
        "physical_qubits",
        "runtime_seconds",
        "logical_error_rate",
        "code_distance",
    }
]

for row in quantity_catalog:
    print(row["quantity"], row["unit"], row["category"])

assert {row["quantity"] for row in quantity_catalog} == {
    "lambda_norm",
    "target_precision",
    "qpe_iterations",
    "toffoli_gates",
    "t_gates",
    "logical_qubits",
    "physical_qubits",
    "runtime_seconds",
    "logical_error_rate",
    "code_distance",
}

lambda_norm energy problem
target_precision energy algorithm
qpe_iterations iterations algorithm
logical_qubits logical qubits logical
toffoli_gates Toffoli gates logical
t_gates T gates logical
physical_qubits physical qubits physical
runtime_seconds seconds physical
logical_error_rate probability physical
code_distance distance architecture


小さなQamomile observableから始めます。実際の化学計算pipelineと同じ形にするためです。Hamiltonianを作る、または読み込み、LCU量を要約して、そのsummaryをFTQC Estimatorへ渡します。下のrescalingは、toy Hamiltonianを大きなactive-space modelの代わりとして使うためのものです。この係数が特定の分子を表すという主張ではありません。

In [5]:
toy_hamiltonian = 0.5 * qm_o.Z(0) + 0.25 * qm_o.X(1) * qm_o.X(2) + 0.125
toy_summary = summarize_pauli_hamiltonian(
    toy_hamiltonian,
    n_spin_orbitals=n_spin_orbitals,
    source="toy_pauli_lcu",
)
scaled_summary = toy_summary.with_lambda_scale(
    sp.Float("2.0e5") / toy_summary.lambda_norm,
    source="scaled_toy_pauli_lcu",
)

assert toy_summary.n_pauli_terms == 2
assert toy_summary.constant == sp.Float("0.125")
assert sp.simplify(scaled_summary.lambda_norm - sp.Float("2.0e5")) == 0

chemistry preprocessingがすでにOpenFermionの`QubitOperator`を生成する場合も、
同じ境界でsummaryを作れます。これにより、electronic-structure toolchainは
Qamomileのcompiler IRの外側に保ちながら、コストを支配するHamiltonian
metadataを保持できます。

In [6]:
openfermion_hamiltonian = (
    QubitOperator("Z0", 0.5)
    + QubitOperator("X1 X2", 0.25)
    + QubitOperator((), 0.125)
)
openfermion_summary = summarize_openfermion_qubit_operator(
    openfermion_hamiltonian,
    n_spin_orbitals=n_spin_orbitals,
    source="openfermion_toy_pauli_lcu",
)

assert openfermion_summary.n_pauli_terms == toy_summary.n_pauli_terms
assert openfermion_summary.lambda_norm == toy_summary.lambda_norm
assert openfermion_summary.constant == toy_summary.constant

## Qubitized QPE Comparison

Qamomileはchemistry factorization costをIRの外側に保ちます。model-driven Estimatorは、Hamiltonian summaryとrepresentation-dependentなone-walk Toffoli costを入力として受け取ります。

```text
qpe_iterations = lambda_norm / precision
toffoli_gates = qpe_iterations * walk_cost_toffoli
```

これによりモデルを明確に保てます。Hamiltonian normalizationを変えること、walk circuitを変えること、physical architectureを変えることは、それぞれ別の設計選択です。

In [7]:
thc_model = ChemistryQPEModel(
    hamiltonian=scaled_summary,
    method=ChemistryQPEMethod.TENSOR_HYPERCONTRACTION,
    walk_cost_toffoli=sp.Integer(4_000),
    description="THC-style scaled toy model",
)
scdf_model = ChemistryQPEModel(
    hamiltonian=scaled_summary.with_lambda_scale(
        sp.Float("0.5"),
        source="SCDF-style scaled toy model",
    ),
    method=ChemistryQPEMethod.SYMMETRY_COMPRESSED_DF,
    walk_cost_toffoli=sp.Integer(4_400),
    second_factor_rank=9,
    description="SCDF-style scaled toy model",
)

thc = estimate_qubitized_chemistry_qpe_from_model(
    thc_model,
    precision=precision,
    cost_model=cost_model,
)
scdf = estimate_qubitized_chemistry_qpe_from_model(
    scdf_model,
    precision=precision,
    cost_model=cost_model,
)

assert scdf.qpe_iterations < thc.qpe_iterations
assert scdf.toffoli_gates < thc.toffoli_gates
assert scdf.resource_values()[FTQCResourceQuantity.CODE_DISTANCE] == 21
assert scdf.to_dict()["architecture_values"]["code_distance"] == "21"

print("THC Toffoli gates:", sp.N(thc.toffoli_gates, 4))
print("SCDF-style Toffoli gates:", sp.N(scdf.toffoli_gates, 4))
print("Toy Pauli terms:", toy_summary.n_pauli_terms)
print("SCDF-style logical qubits:", scdf.logical_qubits)

for row in scdf.to_quantity_table():
    if row["quantity"] in {"qpe_iterations", "toffoli_gates", "physical_qubits"}:
        print(row["label"], row["value"], row["unit"])

qubitized_savings = compare_ftqc_resource_estimates(
    thc,
    scdf,
    quantities=("qpe_iterations", "toffoli_gates", "physical_qubits"),
)
for row in qubitized_savings:
    print(
        row.label,
        "ratio:",
        sp.N(row.ratio, 4),
        "reduction:",
        sp.N(row.reduction, 4),
    )

assert qubitized_savings[0].quantity.value == "qpe_iterations"
assert qubitized_savings[0].ratio == sp.Float("0.5")
assert qubitized_savings[1].ratio == sp.Float("0.55")

qubitized_summary = summarize_ftqc_resource_comparison(
    thc,
    scdf,
    quantities=("qpe_iterations", "toffoli_gates", "physical_qubits"),
)

assert qubitized_summary.smaller[0].quantity == FTQCResourceQuantity.QPE_ITERATIONS
assert qubitized_summary.larger[0].quantity == FTQCResourceQuantity.PHYSICAL_QUBITS

THC Toffoli gates: 5.000e+11
SCDF-style Toffoli gates: 2.750e+11
Toy Pauli terms: 2
SCDF-style logical qubits: 120
Physical qubits 125840 physical qubits
Toffoli gates 275000000000.000 Toffoli gates
QPE iterations 62500000.0000000 iterations
QPE iterations ratio: 0.5000 reduction: 0.5000
Toffoli gates ratio: 0.5500 reduction: 0.4500
Physical qubits ratio: 2.276 reduction: -1.276


同じchemistry modelはblock-encoding contractにも変換できます。将来のloader実装では、chemistry summaryやcompiler IRを変えずに、PREPARE、SELECT、reflection、workspaceのcostを分けてreviewできます。

In [8]:
scdf_block = block_encoding_from_chemistry_model(
    scdf_model,
    prepare_cost_toffoli=sp.Integer(200),
    reflection_cost_toffoli=sp.Integer(50),
    name="scdf_block_contract",
)
scdf_block_estimate = estimate_qubitized_qpe_from_block_encoding(
    scdf_block,
    precision=precision,
    qpe_register_qubits=12,
    cost_model=cost_model,
)

print(scdf_block.to_dict())
assert scdf_block.walk_cost_toffoli == scdf_model.walk_cost_toffoli + 450
assert scdf_block_estimate.target_precision == precision
assert scdf_block_estimate.logical_qubits == scdf_block.logical_qubits + 12

{'name': 'scdf_block_contract', 'system_qubits': '40', 'normalization': '100000.000000000', 'select_cost_toffoli': '4400', 'prepare_cost_toffoli': '200', 'reflection_cost_toffoli': '50', 'ancilla_qubits': '80', 'logical_qubits': '120', 'walk_cost_toffoli': '4850', 'references': [{'key': 'arXiv:1902.02134', 'title': 'Qubitization of Arbitrary Basis Quantum Chemistry Leveraging Sparsity and Low Rank Factorization', 'url': 'https://arxiv.org/abs/1902.02134', 'note': 'Provides sparse and low-rank qubitized chemistry cost models that motivate representation-specific logical-resource scaling.'}, {'key': 'arXiv:2403.03502', 'title': 'Reducing the runtime of fault-tolerant quantum simulations in chemistry through symmetry-compressed double factorization', 'url': 'https://arxiv.org/abs/2403.03502', 'note': 'Introduces symmetry-compressed double factorization and reports Hamiltonian 1-norm and Toffoli-count reductions.'}, {'key': 'arXiv:2412.01338', 'title': 'Simultaneously optimizing symmetry s

## Early-FTQC Trotter QPE

Early-FTQCの提案では、量子ビット数とdepthが強く制限される場合、qubitized walksよりも浅いsingle-ancilla QPEとPauli rotationsが好まれることがあります。下のunitary-weight concentration factorは、コストを支配するeffective weightを下げるspectrally invariantなHamiltonian transformationを表します。

In [9]:
plain_trotter = estimate_single_ancilla_trotter_qpe_from_hamiltonian(
    scaled_summary,
    precision=precision,
    trotter_steps_per_sample=8,
    samples=128,
    cost_model=cost_model,
)

uwc_trotter = estimate_single_ancilla_trotter_qpe_from_hamiltonian(
    scaled_summary,
    precision=precision,
    trotter_steps_per_sample=8,
    samples=128,
    unitary_weight_factor=sp.Float("0.1"),
    randomized_compilation_factor=sp.Float("0.5"),
    rotation_synthesis_t_gates=3,
    cost_model=cost_model,
)

assert uwc_trotter.qpe_iterations < plain_trotter.qpe_iterations
assert uwc_trotter.logical_depth < plain_trotter.logical_depth

print("Plain Trotter QPE depth proxy:", sp.N(plain_trotter.logical_depth, 4))
print("UWC-style Trotter QPE depth proxy:", sp.N(uwc_trotter.logical_depth, 4))
print("UWC-style T gates:", sp.N(uwc_trotter.t_gates, 4))

trotter_savings = compare_ftqc_resource_estimates(
    plain_trotter,
    uwc_trotter,
    quantities=("qpe_iterations", "logical_depth"),
)
for row in trotter_savings:
    print(
        row.label,
        "ratio:",
        sp.N(row.ratio, 4),
        "reduction:",
        sp.N(row.reduction, 4),
    )

assert trotter_savings[0].ratio == sp.Float("0.1")
assert trotter_savings[1].ratio == sp.Float("0.05")

Plain Trotter QPE depth proxy: 2.560e+11
UWC-style Trotter QPE depth proxy: 1.280e+10
UWC-style T gates: 3.840e+10
QPE iterations ratio: 0.1000 reduction: 0.9000
Logical depth ratio: 0.05000 reduction: 0.9500


## Result

推定結果を小さな表にまとめます。重要なのは、各列が別々の設計上の意味を持つことです。Hamiltonian representationの変更は`qpe_iterations`とper-step costに効くべきで、hardware modelの変更は`physical_qubits`とruntimeに効くべきです。

In [10]:
rows = [
    ("THC qubitized QPE", thc),
    ("SCDF-style qubitized QPE", scdf),
    ("Plain Trotter QPE", plain_trotter),
    ("UWC-style Trotter QPE", uwc_trotter),
]

for name, estimate in rows:
    print(
        name,
        {
            "logical_qubits": sp.N(estimate.logical_qubits, 4),
            "physical_qubits": sp.N(estimate.physical_qubits, 4),
            "qpe_iterations": sp.N(estimate.qpe_iterations, 4),
            "toffoli_gates": sp.N(estimate.toffoli_gates, 4),
            "t_gates": sp.N(estimate.t_gates, 4),
            "runtime_seconds": sp.N(estimate.runtime_seconds, 4),
        },
    )

assert scdf.physical_qubits > thc.physical_qubits
assert uwc_trotter.physical_qubits == plain_trotter.physical_qubits

THC qubitized QPE {'logical_qubits': 40.00, 'physical_qubits': 5.528e+4, 'qpe_iterations': 1.250e+8, 'toffoli_gates': 5.000e+11, 't_gates': 0, 'runtime_seconds': 5.250e+5}
SCDF-style qubitized QPE {'logical_qubits': 120.0, 'physical_qubits': 1.258e+5, 'qpe_iterations': 6.250e+7, 'toffoli_gates': 2.750e+11, 't_gates': 0, 'runtime_seconds': 2.888e+5}
Plain Trotter QPE {'logical_qubits': 41.00, 'physical_qubits': 5.616e+4, 'qpe_iterations': 1.250e+8, 'toffoli_gates': 0, 't_gates': 2.560e+11, 'runtime_seconds': 2.688e+5}
UWC-style Trotter QPE {'logical_qubits': 41.00, 'physical_qubits': 5.616e+4, 'qpe_iterations': 1.250e+7, 'toffoli_gates': 0, 't_gates': 3.840e+10, 'runtime_seconds': 2.016e+4}


## Summary

このノートブックでは、次のことを確認しました。

- FTQC化学計算のリソース量を、Hamiltonian normalization、QPE iterations、non-Clifford counts、論理量子ビット、物理量子ビット、runtime proxiesに分けて扱いました。
- Qamomile IRをbackend-specificなchemistry loading circuitsへloweringせずに、qubitized QPE representationsを比較しました。
- logical failure budgetからsurface-code distanceを選び、logical resourceをphysical resourceへliftしました。
- code distanceなどのarchitecture quantityを各resource estimateに残し、後続のreportでauditできるようにしました。
- chemistry QPE modelをblock-encoding contractへ変換し、PREPARE、SELECT、reflection、workspace costを分けてreviewできる形にしました。
- unitary-weight concentration factorを、early-FTQC single-ancilla Trotter QPEのcost-driver reductionとしてモデル化する方法を示しました。